# Build Dataset v2

Build `kaya-go/moku-v2` by combining:
1. **Corrected v1** — Real images with human-corrected board_corner annotations
2. **Synthetic data** — Generated goban images with perfect annotations (diverse backgrounds, 3D perspective, lighting)

**Training strategy**: Two-stage (synthetic pre-train → real fine-tune), so we push them as separate configs in the same HF dataset.

**Categories:**
| ID | Name | Description |
|-----|------|------|
| 0 | black_stone | Individual black stone |
| 1 | white_stone | Individual white stone |
| 2 | board_corner | Board corner point (small bbox at each corner) |

See [docs/dataset.md](../docs/dataset.md) for full details.

In [1]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from datasets import Image as HFImage

from moku.dataset import (
    CATEGORIES,
    ID_TO_CATEGORY,
    apply_corner_corrections,
    build_dataset,
    compute_split_stats,
)
from moku.synthetic import generate_synthetic_sample

## Step 1: Build Corrected Real Dataset

Rebuild v1 from raw data and apply human-corrected corner annotations from the annotator.

In [2]:
# Rebuild from raw data
RAW_DATA_DIR = Path("/Users/hadim/Data/moku/raw")
real_dataset = build_dataset(RAW_DATA_DIR)

# Apply corner corrections
corrections_path = Path("data/annotate/corrected.json")
with open(corrections_path) as f:
    corrections = json.load(f)

print(f"Loaded {len(corrections)} corrected images")
real_dataset = apply_corner_corrections(real_dataset, corrections)
print(real_dataset)

  go_game_v10: 249 images
  go_chess: 236 images
  Total pooled: 485 images
  train: 382 images (153 base) [go_chess: 195, go_game_v10: 187] — 370 with 4 corners
  validation: 53 images (20 base) [go_chess: 25, go_game_v10: 28] — 50 with 4 corners
  test: 50 images (20 base) [go_chess: 16, go_game_v10: 34] — 47 with 4 corners
Loaded 454 corrected images


Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 382
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 53
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 50
    })
})


## Step 2: Generate Synthetic Dataset

Generate synthetic goban images with diverse backgrounds, 3D perspective, and lighting effects.
Split into train/validation/test (2500/500/500). Stone count is biased toward dense boards via a beta distribution.

In [3]:
import random as _rng

from tqdm.auto import tqdm

SYNTHETIC_SPLITS = {"train": 2500, "validation": 500, "test": 500}

# Beta(2, 1.2) skews toward dense boards (~62% mean occupancy).
BETA_A, BETA_B = 2.0, 1.2
MIN_STONES = 3
MAX_OCCUPANCY = 0.80


def generate_split(n_samples: int, seed: int = 42) -> Dataset:
    """Generate a synthetic dataset split."""
    _rng.seed(seed)
    np.random.seed(seed)

    rows = []
    for i in tqdm(range(n_samples), desc="Generating"):
        board_size = _rng.choices([9, 13, 19], weights=[0.15, 0.15, 0.70])[0]
        max_stones = int(board_size**2 * MAX_OCCUPANCY)
        frac = _rng.betavariate(BETA_A, BETA_B)
        n_stones = max(MIN_STONES, int(frac * max_stones))
        persp = _rng.uniform(0.0, 0.12)

        image, annotation = generate_synthetic_sample(
            board_size=board_size,
            image_size=640,
            n_stones=n_stones,
            perspective_strength=persp,
        )

        rows.append(
            {
                "image": image,
                "image_id": i,
                "width": annotation["width"],
                "height": annotation["height"],
                "source_dataset": "synthetic",
                "objects": annotation["objects"],
            }
        )

    return Dataset.from_list(rows).cast_column("image", HFImage())


synth_splits = {}
for split_name, n in SYNTHETIC_SPLITS.items():
    print(f"Generating {split_name}: {n} samples...")
    synth_splits[split_name] = generate_split(n, seed=42 + hash(split_name) % 1000)

synthetic_dataset = DatasetDict(synth_splits)
print(f"\nSynthetic dataset: {synthetic_dataset}")

Generating train: 2500 samples...


Generating:   0%|          | 0/2500 [00:00<?, ?it/s]

Generating validation: 500 samples...


Generating:   0%|          | 0/500 [00:00<?, ?it/s]

Generating test: 500 samples...


Generating:   0%|          | 0/500 [00:00<?, ?it/s]


Synthetic dataset: DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 500
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 500
    })
})


## Step 3: Dataset Statistics

Review both real (corrected) and synthetic datasets before pushing.

In [4]:
stats_rows = []
for label, ds_dict in [("real", real_dataset), ("synthetic", synthetic_dataset)]:
    for split_name, split_ds in ds_dict.items():
        stats = compute_split_stats(split_ds)
        stats_rows.append(
            {
                "dataset": label,
                "split": split_name,
                "images": stats["num_images"],
                "total_objects": stats["total_objects"],
                "avg_objects_per_image": round(stats["avg_objects_per_image"], 1),
            }
        )

display(pd.DataFrame(stats_rows))

# Per-category breakdown
cat_rows = []
for label, ds_dict in [("real", real_dataset), ("synthetic", synthetic_dataset)]:
    for split_name, split_ds in ds_dict.items():
        stats = compute_split_stats(split_ds)
        for cat_id, count in stats["category_counts"].most_common():
            cat_rows.append(
                {
                    "dataset": label,
                    "split": split_name,
                    "category": ID_TO_CATEGORY[cat_id],
                    "count": count,
                }
            )

display(pd.DataFrame(cat_rows))

,dataset,split,images,total_objects,avg_objects_per_image
0,real,train,382,25787,67.5
1,real,validation,53,2378,44.9
2,real,test,50,3179,63.6
3,synthetic,train,2500,372591,149.0
4,synthetic,validation,500,74821,149.6
5,synthetic,test,500,73791,147.6


,dataset,split,category,count
0,real,train,black_stone,12902
1,real,train,white_stone,11364
2,real,train,board_corner,1521
3,real,validation,black_stone,1205
4,real,validation,white_stone,962
5,real,validation,board_corner,211
6,real,test,black_stone,1602
7,real,test,white_stone,1377
8,real,test,board_corner,200
9,synthetic,train,white_stone,181513


## Browse Dataset

Interactive browser to navigate samples and inspect annotations.

In [6]:
from moku.viz import browse_dataset

browse_dataset(real_dataset)
# browse_dataset(synthetic_dataset)

## Step 4: Push to Hugging Face Hub

Push both datasets as separate configs in `kaya-go/moku-v2`:
- `real` config — corrected real images (used in stage 2 fine-tuning)
- `synthetic` config — generated images (used in stage 1 pre-training)

In [7]:
HF_DATASET_V2 = "kaya-go/moku-v2"

# Push real dataset as "real" config
real_dataset.push_to_hub(HF_DATASET_V2, config_name="real", private=False)
print(f"Real dataset pushed to {HF_DATASET_V2} (config=real)")

# Push synthetic dataset as "synthetic" config
synthetic_dataset.push_to_hub(HF_DATASET_V2, config_name="synthetic", private=False)
print(f"Synthetic dataset pushed to {HF_DATASET_V2} (config=synthetic)")

print(f"\nDataset available at: https://huggingface.co/datasets/{HF_DATASET_V2}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/879 [00:00<?, ?B/s]

Synthetic dataset pushed to kaya-go/moku-v2 (config=synthetic)

Dataset available at: https://huggingface.co/datasets/kaya-go/moku-v2
